# A.8.1 Defined variables

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

In some nonlinear models, it is convenient to define named values that contribute somehow to
the constraints or objectives. For example,

```ampl
set A;
var v {A};
var w {A};
subject to C {i in A}: w[i] = vexpr;
```

where vexpr is an expression involving the variables v.

As things stand, the members of C are constraints, and we have turned an unconstrained problem
into a constrained one, which may not be a good idea. Setting option substout to 1
instructs AMPL to eliminate the collection of constraints C. AMPL does so by telling solvers that
the constraints define the variables on their left-hand sides, so that, in effect, these defined variables
become named common subexpressions.

When option substout is 1, a constraint such as C that provides a definition for a defined
variable is called a defining constraint. AMPL decides which variables are defined variables by
scanning the constraints once, in order of their appearance in the model. A variable is eligible to
become a defined variable only if its declaration imposes no constraints on it, such as integrality or
bounds. Once a variable has appeared on the right-hand side of a defining constraint, it is no
longer eligible to be a defined variable — without this restriction, AMPL might have to solve
implicit equations. Once a variable has been recognized as a defined variable, its subsequent
appearance on the left-hand side of what would otherwise be a defining constraint results in the
constraint being treated as an ordinary constraint.

Some solvers give special treatment to linear variables because their higher derivatives vanish.
For such solvers, it may be helpful to treat linear defined variables specially. Otherwise, variables
involved in the right-hand side of the equation for a defined variable appear to solvers as nonlinear
variables, even if they are used only linearly in the right-hand side. By doing Gaussian elimination
rather than conveying linear variable definitions explicitly, AMPL can arrange for solvers to see
such right-hand variables as linear variables. This often causes fill-in, i.e., makes the problem less
sparse, but it may give the solvers a more accurate view of the problem. When option linelim
has its default value 1, AMPL treats linear defined variables in this special way; when option linelim
is 0, AMPL treats all defined variables alike.
A variable declaration may have a phrase of the form = expr, where expr is an expression
involving variables. Such a phrase indicates that the variable is to be defined with the value expr.
Such defining declarations allow some models to be written more compactly.

Recognizing defined variables is not always a good idea — it leads to a problem in fewer variables,
but one that may be more nonlinear and that may be more expensive to solve because of loss
of sparsity. By using defining constraints (instead of using defining variable declarations) and
translating and solving a problem both with $substout = 0 and with $substout = 1, one can
see whether recognizing defined variables is worthwhile. On the other hand, if recognizing a
defined variable seems clearly worthwhile, defining it in its declaration is often more convenient
than providing a separate defining constraint; in particular, if all defined variables are defined in
their declarations, one need not worry about $substout.

One restriction on defining declarations is that subscripted variables must be defined before
they are used.